<a href="https://colab.research.google.com/github/sejiiii2/finsight/blob/main/05_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/anikanb-32/musicandmemory.git
%cd musicandmemory

Cloning into 'musicandmemory'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 137 (delta 74), reused 42 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 56.03 KiB | 1.22 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/musicandmemory/musicandmemory


In [6]:
import json
from src.tuning import tune_variant_b, tune_variant_c
from src.retrieval import load_retrieval_system

faiss_index, bm25, df = load_retrieval_system(
    "data/index/songs.index",
    "data/index/bm25.pkl",
    "data/processed/knowledge_base.csv"
)

with open("data/processed/val_profiles.json") as f:
    val_profiles = json.load(f)

# Tune Variant B
print("=" * 60)
print("TUNING VARIANT B")
print("=" * 60)
tuning_b = tune_variant_b(val_profiles, faiss_index, bm25, df)
tuning_b.to_csv("outputs/tuning_variant_b.csv", index=False)

# Tune Variant C
print("=" * 60)
print("TUNING VARIANT C")
print("=" * 60)
tuning_c = tune_variant_c(val_profiles, faiss_index, bm25, df)
tuning_c.to_csv("outputs/tuning_variant_c.csv", index=False)

# Identify best configurations
best_b = tuning_b.sort_values("avg_overall_llm", ascending=False).iloc[0]
best_c = tuning_c.sort_values("avg_overall_llm", ascending=False).iloc[0]

print(f"\nBest Variant B config: method={best_b['method']}, k={int(best_b['k'])}")
print(f"Best Variant C config: method={best_c['method']}, k={int(best_c['k'])}")

# Save best configs
best_configs = {
    "variant_b": {"method": best_b["method"], "k": int(best_b["k"])},
    "variant_c": {"method": best_c["method"], "k": int(best_c["k"])},
}
with open("outputs/best_configs.json", "w") as f:
    json.dump(best_configs, f, indent=2)

ModuleNotFoundError: No module named 'faiss'